# Employee Attrition Prediction

This notebook builds a complete classification pipeline for employee attrition using the CSV in the current working directory. It covers data loading, cleaning, exploratory analysis, feature engineering, encoding, feature selection, model training, tuning, SHAP explainability, saving, and inference. The target column is `Attrition`.

In [1]:
import importlib
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

for module_name, package_name in {'sklearn': 'scikit-learn', 'xgboost': 'xgboost', 'catboost': 'catboost', 'shap': 'shap'}.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name])

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from catboost import CatBoostClassifier
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titleweight'] = 'bold'
print('Libraries imported successfully.')

ModuleNotFoundError: No module named 'seaborn'

## Section 3 - Data Loading

The dataset is auto-detected from the current directory so the notebook stays portable.

## Section 4 - Data Understanding

This section inspects the raw dataframe before cleaning.

In [ ]:
csvs = sorted(Path.cwd().glob('*.csv'))
if not csvs:
    raise FileNotFoundError('No CSV file found in the current working directory.')

data_path = next((p for p in csvs if 'attrition' in p.name.lower()), csvs[0])
df = pd.read_csv(data_path)
df.columns = df.columns.str.strip()
target_col = 'Attrition'

print(f'Loaded: {data_path.name}')
print(f'Shape: {df.shape}')
                "models = {",
                "    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),",
                "    'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),",
                "    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced_subsample'),",
                "    'XGBoost': XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=-1, eval_metric='logloss'),",
                "    'CatBoost': CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6, random_seed=42, verbose=0, allow_writing_files=False)",
                "}",
                "",
                "trained, rows, roc_data = {}, [], {}",
                "for name, model in models.items():",
                "    pipe = make_pipeline(model).fit(X_train_raw, y_train)",
                "    pred = pipe.predict(X_test_raw)",
                "    p = proba(pipe, X_test_raw)",
                "    trained[name], roc_data[name] = pipe, p",
                "    rows.append({'model': name, 'accuracy': accuracy_score(y_test, pred), 'precision': precision_score(y_test, pred, zero_division=0), 'recall': recall_score(y_test, pred, zero_division=0), 'f1_score': f1_score(y_test, pred, zero_division=0), 'roc_auc': roc_auc_score(y_test, p)})",
                "",
                "results_df = pd.DataFrame(rows).sort_values(['roc_auc', 'f1_score'], ascending=False).reset_index(drop=True)",
                "display(results_df)",
                "best_model_name = results_df.iloc[0]['model']",
                "best_pipe = trained[best_model_name]",
                "best_pred = best_pipe.predict(X_test_raw)",
                "best_proba = roc_data[best_model_name]",
                "",
                "fig, ax = plt.subplots(figsize=(7, 6))",
                "sns.heatmap(confusion_matrix(y_test, best_pred), annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)",
                "ax.set_title(f'Confusion Matrix: {best_model_name}')",
                "plt.tight_layout(); plt.show()",
                "",
                "plt.figure(figsize=(8, 6))",
                "for name, p in roc_data.items():",
                "    fpr, tpr, _ = roc_curve(y_test, p)",
                "    auc = results_df.loc[results_df['model'] == name, 'roc_auc'].iloc[0]",
                "    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc:.3f})')",
                "plt.plot([0, 1], [0, 1], 'k--')",
                "plt.title('ROC Curves')",
                "plt.xlabel('False Positive Rate')",
                "plt.ylabel('True Positive Rate')",
                "plt.legend(loc='lower right')",
                "plt.tight_layout(); plt.show()",
                "",
                "search_spaces = {",
                "    'Logistic Regression': {'model__C': [0.01, 0.1, 1, 10], 'model__solver': ['liblinear'], 'model__penalty': ['l1', 'l2']},",
                "    'Decision Tree': {'model__max_depth': [None, 3, 5, 8, 12], 'model__min_samples_split': [2, 5, 10], 'model__min_samples_leaf': [1, 2, 4], 'model__criterion': ['gini', 'entropy']},",
                "    'Random Forest': {'model__n_estimators': [200, 300, 500], 'model__max_depth': [None, 5, 10, 15], 'model__min_samples_split': [2, 5, 10], 'model__min_samples_leaf': [1, 2, 4], 'model__max_features': ['sqrt', 'log2', None]},",
                "    'XGBoost': {'model__n_estimators': [200, 300, 500], 'model__max_depth': [3, 4, 5, 6], 'model__learning_rate': [0.01, 0.05, 0.1], 'model__subsample': [0.7, 0.85, 1.0], 'model__colsample_bytree': [0.7, 0.85, 1.0], 'model__min_child_weight': [1, 3, 5], 'model__gamma': [0, 0.1, 0.2]},",
                "    'CatBoost': {'model__iterations': [200, 300, 500], 'model__depth': [4, 6, 8, 10], 'model__learning_rate': [0.01, 0.05, 0.1], 'model__l2_leaf_reg': [1, 3, 5, 7], 'model__bagging_temperature': [0, 0.5, 1.0]}",
                "}",
                "",
                "search = RandomizedSearchCV(best_pipe, search_spaces[best_model_name], n_iter=10, scoring='roc_auc', cv=StratifiedKFold(5, shuffle=True, random_state=42), random_state=42, n_jobs=-1, refit=True)",
                "search.fit(X_train_raw, y_train)",
                "tuned_pipe = search.best_estimator_",
                "tuned_pred = tuned_pipe.predict(X_test_raw)",
                "tuned_proba = proba(tuned_pipe, X_test_raw)",
                "tune_df = pd.DataFrame([",
                "    {'model': f'Baseline {best_model_name}', 'accuracy': accuracy_score(y_test, best_pred), 'precision': precision_score(y_test, best_pred, zero_division=0), 'recall': recall_score(y_test, best_pred, zero_division=0), 'f1_score': f1_score(y_test, best_pred, zero_division=0), 'roc_auc': roc_auc_score(y_test, best_proba)},",
                "    {'model': f'Tuned {best_model_name}', 'accuracy': accuracy_score(y_test, tuned_pred), 'precision': precision_score(y_test, tuned_pred, zero_division=0), 'recall': recall_score(y_test, tuned_pred, zero_division=0), 'f1_score': f1_score(y_test, tuned_pred, zero_division=0), 'roc_auc': roc_auc_score(y_test, tuned_proba)}",
                "]).sort_values('roc_auc', ascending=False).reset_index(drop=True)",
                "display(search.best_params_)",
                "display(tune_df)",
                "",
                "final_pipe = tuned_pipe if tune_df.iloc[0]['model'].startswith('Tuned') else best_pipe",
                "final_name = tune_df.iloc[0]['model']",
                "final_pred = final_pipe.predict(X_test_raw)",
                "final_proba = proba(final_pipe, X_test_raw)",
                "final_metrics = pd.DataFrame([{'model': final_name, 'accuracy': accuracy_score(y_test, final_pred), 'precision': precision_score(y_test, final_pred, zero_division=0), 'recall': recall_score(y_test, final_pred, zero_division=0), 'f1_score': f1_score(y_test, final_pred, zero_division=0), 'roc_auc': roc_auc_score(y_test, final_proba)}])",
                "display(final_metrics)",
                "",
                "# Robust feature-name extraction for SHAP and inspection",
                "prep = final_pipe.named_steps.get('preprocessor')",
                "sel = final_pipe.named_steps.get('selector')",
                "clf = final_pipe.named_steps.get('model')",
                "# Try to get feature names from preprocessor; fallback to manual construction",
                "try:",
                "    prep_feature_names = prep.get_feature_names_out()",
                "except Exception:",
                "    num_feats = [f for f in X_train_raw.columns if np.issubdtype(X_train_raw[f].dtype, np.number)]",
                "    cat_feats = [f for f in X_train_raw.columns if f not in num_feats]",
                "    try:",
                "        ohe = prep.named_transformers_['cat'].named_steps.get('ohe')",
                "        cat_names = list(ohe.get_feature_names_out(cat_feats)) if ohe is not None else cat_feats",
                "    except Exception:",
                "        cat_names = cat_feats",
                "    prep_feature_names = np.array(list(num_feats) + list(cat_names))",
                "# Apply selector to get selected names if selector exists",
                "try:",
                "    support = sel.get_support()",
                "    sel_names = np.array(prep_feature_names)[support]",
                "except Exception:",
                "    sel_names = prep_feature_names",
                "# Prepare SHAP input matrix safely",
                "try:",
                "    X_sel = sel.transform(prep.transform(X_train_raw)) if sel is not None else prep.transform(X_train_raw)",
                "    sample_idx = np.random.RandomState(42).choice(X_sel.shape[0], size=min(200, X_sel.shape[0]), replace=False)",
                "    X_shap = pd.DataFrame(X_sel[sample_idx], columns=sel_names)",
                "except Exception:",
                "    X_shap = None",
                "# SHAP explanation with graceful fallback",
                "if X_shap is not None and clf is not None:",
                "    try:",
                "        if clf.__class__.__name__ in ['RandomForestClassifier', 'DecisionTreeClassifier', 'XGBClassifier', 'CatBoostClassifier']:",
                "            explainer = shap.TreeExplainer(clf)",
                "            sv = explainer.shap_values(X_shap)",
                "            sv = sv[1] if isinstance(sv, list) else sv",
                "        else:",
                "            explainer = shap.Explainer(clf, X_shap)",
                "            out = explainer(X_shap)",
                "            sv = out.values if hasattr(out, 'values') else out",
                "        shap.summary_plot(sv, X_shap, show=False)",
                "        plt.tight_layout(); plt.show()",
                "        shap.summary_plot(sv, X_shap, plot_type='bar', show=False)",
                "        plt.tight_layout(); plt.show()",
                "    except Exception as e:",
                "        print('SHAP plotting failed:', e)",
                "        # Fallback: show feature importances if available",
                "        try:",
                "            if hasattr(clf, 'feature_importances_'):",
                "                fi = pd.Series(clf.feature_importances_, index=sel_names).sort_values(ascending=False)",
                "                fi.head(20).plot(kind='barh')",
                "                plt.title('Feature importances (fallback)')",
                "                plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()",
                "        except Exception as e2:
                "            print('Fallback feature importance plotting failed:', e2)",
                "",
                "# Save model artifact",
                "env_models = Path('models')",
                "env_models.mkdir(exist_ok=True)",
                "model_path = env_models / 'employee_attrition_model.pkl'",
                "joblib.dump(final_pipe, model_path)",
                "loaded_model = joblib.load(model_path)",
                "sample_rows = X_test_raw.sample(n=min(5, len(X_test_raw)), random_state=42)",
                "demo = pd.DataFrame({",
                "    'Actual_Attrition': y_test.loc[sample_rows.index].map({0: 'No', 1: 'Yes'}),",
                "    'Predicted_Attrition': pd.Series(loaded_model.predict(sample_rows), index=sample_rows.index).map({0: 'No', 1: 'Yes'}),",
                "    'Probability_Attrition_Yes': np.round(proba(loaded_model, sample_rows), 4)",
                "})",
                "print(f'Model saved to: {model_path.resolve()}')",
                "display(demo)"
## Section 9 - Feature Selection

A stratified train/test split is created, then correlations, mutual information, and random forest importance are used to rank features on the training fold only.

## Section 10 - Data Preprocessing

The modeling pipeline uses the 80/20 stratified split above. Imputation, scaling, encoding, and feature selection are all wrapped into reusable pipelines to avoid leakage.

In [ ]:
df_fe = df_clean.copy()
df_fe['AgeCategory'] = pd.cut(df_fe['Age'], bins=[0, 30, 40, 50, np.inf], labels=['Young', 'Early-Career', 'Mid-Career', 'Senior'])
df_fe['IncomeBand'] = pd.cut(df_fe['MonthlyIncome'], bins=[-np.inf, 3000, 6000, 10000, np.inf], labels=['Low', 'Moderate', 'High', 'Very High'])
df_fe['ExperienceGroup'] = pd.cut(df_fe['TotalWorkingYears'], bins=[-np.inf, 5, 10, 20, np.inf], labels=['Entry', 'Developing', 'Established', 'Veteran'])
df_fe['TenureBucket'] = pd.cut(df_fe['YearsAtCompany'], bins=[-np.inf, 2, 5, 10, np.inf], labels=['New Joiner', 'Growing', 'Long-Term', 'Core Talent'])
display(df_fe[['AgeCategory', 'IncomeBand', 'ExperienceGroup', 'TenureBucket']].head())

y = df_fe[target_col].astype(str).str.strip().str.lower().map({'no': 0, 'yes': 1})
if y.isna().any():
    raise ValueError('Attrition must contain Yes/No values only.')
X = df_fe.drop(columns=[target_col])
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_features = [c for c in X.columns if c not in categorical_features]

def make_ohe():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

def make_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', make_ohe())]), categorical_features)
    ])

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
preprocessor = make_preprocessor()
X_train_proc = preprocessor.fit_transform(X_train_raw)
X_test_proc = preprocessor.transform(X_test_raw)
feature_names = preprocessor.get_feature_names_out()

train_num = X_train_raw.select_dtypes(include=[np.number]).copy(); train_num[target_col] = y_train.values
corr_rank = train_num.corr(numeric_only=True)[target_col].drop(target_col).sort_values(key=np.abs, ascending=False)
display(corr_rank.head(10).to_frame('correlation'))

mi_ranker = SelectKBest(mutual_info_classif, k=min(20, X_train_proc.shape[1]))
mi_ranker.fit(X_train_proc, y_train)
mi_scores = pd.Series(mi_ranker.scores_, index=feature_names).sort_values(ascending=False)
display(mi_scores.head(15).to_frame('mutual_information'))

rf_ranker = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced_subsample')
rf_ranker.fit(X_train_proc, y_train)
rf_scores = pd.Series(rf_ranker.feature_importances_, index=feature_names).sort_values(ascending=False)
display(rf_scores.head(15).to_frame('feature_importance'))

rank_df = pd.DataFrame({'feature': feature_names, 'mi': mi_scores.reindex(feature_names).fillna(0).values, 'rf': rf_scores.reindex(feature_names).fillna(0).values})
rank_df['rank'] = rank_df[['mi', 'rf']].rank(ascending=False).mean(axis=1)
rank_df = rank_df.sort_values(['rank', 'feature']).reset_index(drop=True)
best_k = min(20, len(rank_df))
selected_features = rank_df.head(best_k)['feature'].tolist()
display(rank_df.head(best_k))

print('Train shape:', X_train_raw.shape)
print('Test shape:', X_test_raw.shape)
print('Processed train shape:', X_train_proc.shape)
print('Selected features:', best_k)

def make_pipeline(model):
    return Pipeline([('preprocessor', make_preprocessor()), ('selector', SelectKBest(mutual_info_classif, k=best_k)), ('model', model)])

def proba(model, data):
    return model.predict_proba(data)[:, 1] if hasattr(model, 'predict_proba') else 1 / (1 + np.exp(-model.decision_function(data)))

## Section 11 - Model Training

Five models are trained on the same preprocessing pipeline: Logistic Regression, Decision Tree, Random Forest, XGBoost, and CatBoost.

## Section 12 - Model Evaluation

Each trained model is evaluated with Accuracy, Precision, Recall, F1 Score, and ROC-AUC. Confusion matrices and ROC curves are generated for comparison.

## Section 13 - Hyperparameter Tuning

The best baseline model is tuned with RandomizedSearchCV using a compact search space for practical execution.

## Section 14 - Final Model Selection

The best model is selected from holdout performance after tuning.

## Section 15 - SHAP Explainability

SHAP is used to explain the final model and identify the main drivers of attrition.

## Section 16 - Save Model

The final fitted pipeline is saved with Joblib to `models/employee_attrition_model.pkl`.

## Section 17 - Prediction Demo

A few holdout records are scored to show the predicted class and probability.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced_subsample'),
    'XGBoost': XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=-1, eval_metric='logloss'),
    'CatBoost': CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6, random_seed=42, verbose=0, allow_writing_files=False)
}

trained, rows, roc_data = {}, [], {}
for name, model in models.items():
    pipe = make_pipeline(model).fit(X_train_raw, y_train)
    pred = pipe.predict(X_test_raw)
    p = proba(pipe, X_test_raw)
    trained[name], roc_data[name] = pipe, p
    rows.append({'model': name, 'accuracy': accuracy_score(y_test, pred), 'precision': precision_score(y_test, pred, zero_division=0), 'recall': recall_score(y_test, pred, zero_division=0), 'f1_score': f1_score(y_test, pred, zero_division=0), 'roc_auc': roc_auc_score(y_test, p)})

results_df = pd.DataFrame(rows).sort_values(['roc_auc', 'f1_score'], ascending=False).reset_index(drop=True)
display(results_df)
best_model_name = results_df.iloc[0]['model']
best_pipe = trained[best_model_name]
best_pred = best_pipe.predict(X_test_raw)
best_proba = roc_data[best_model_name]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(confusion_matrix(y_test, best_pred), annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
ax.set_title(f'Confusion Matrix: {best_model_name}')
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 6))
for name, p in roc_data.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    auc = results_df.loc[results_df['model'] == name, 'roc_auc'].iloc[0]
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curves')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.tight_layout(); plt.show()

search_spaces = {
    'Logistic Regression': {'model__C': [0.01, 0.1, 1, 10], 'model__solver': ['liblinear'], 'model__penalty': ['l1', 'l2']},
    'Decision Tree': {'model__max_depth': [None, 3, 5, 8, 12], 'model__min_samples_split': [2, 5, 10], 'model__min_samples_leaf': [1, 2, 4], 'model__criterion': ['gini', 'entropy']},
    'Random Forest': {'model__n_estimators': [200, 300, 500], 'model__max_depth': [None, 5, 10, 15], 'model__min_samples_split': [2, 5, 10], 'model__min_samples_leaf': [1, 2, 4], 'model__max_features': ['sqrt', 'log2', None]},
    'XGBoost': {'model__n_estimators': [200, 300, 500], 'model__max_depth': [3, 4, 5, 6], 'model__learning_rate': [0.01, 0.05, 0.1], 'model__subsample': [0.7, 0.85, 1.0], 'model__colsample_bytree': [0.7, 0.85, 1.0], 'model__min_child_weight': [1, 3, 5], 'model__gamma': [0, 0.1, 0.2]},
    'CatBoost': {'model__iterations': [200, 300, 500], 'model__depth': [4, 6, 8, 10], 'model__learning_rate': [0.01, 0.05, 0.1], 'model__l2_leaf_reg': [1, 3, 5, 7], 'model__bagging_temperature': [0, 0.5, 1.0]}
}

search = RandomizedSearchCV(best_pipe, search_spaces[best_model_name], n_iter=10, scoring='roc_auc', cv=StratifiedKFold(5, shuffle=True, random_state=42), random_state=42, n_jobs=-1, refit=True)
search.fit(X_train_raw, y_train)
tuned_pipe = search.best_estimator_
tuned_pred = tuned_pipe.predict(X_test_raw)
tuned_proba = proba(tuned_pipe, X_test_raw)
tune_df = pd.DataFrame([
    {'model': f'Baseline {best_model_name}', 'accuracy': accuracy_score(y_test, best_pred), 'precision': precision_score(y_test, best_pred, zero_division=0), 'recall': recall_score(y_test, best_pred, zero_division=0), 'f1_score': f1_score(y_test, best_pred, zero_division=0), 'roc_auc': roc_auc_score(y_test, best_proba)},
    {'model': f'Tuned {best_model_name}', 'accuracy': accuracy_score(y_test, tuned_pred), 'precision': precision_score(y_test, tuned_pred, zero_division=0), 'recall': recall_score(y_test, tuned_pred, zero_division=0), 'f1_score': f1_score(y_test, tuned_pred, zero_division=0), 'roc_auc': roc_auc_score(y_test, tuned_proba)}
]).sort_values('roc_auc', ascending=False).reset_index(drop=True)
display(search.best_params_)
display(tune_df)

final_pipe = tuned_pipe if tune_df.iloc[0]['model'].startswith('Tuned') else best_pipe
final_name = tune_df.iloc[0]['model']
final_pred = final_pipe.predict(X_test_raw)
final_proba = proba(final_pipe, X_test_raw)
final_metrics = pd.DataFrame([{'model': final_name, 'accuracy': accuracy_score(y_test, final_pred), 'precision': precision_score(y_test, final_pred, zero_division=0), 'recall': recall_score(y_test, final_pred, zero_division=0), 'f1_score': f1_score(y_test, final_pred, zero_division=0), 'roc_auc': roc_auc_score(y_test, final_proba)}])
display(final_metrics)

prep = final_pipe.named_steps['preprocessor']
sel = final_pipe.named_steps['selector']
clf = final_pipe.named_steps['model']
X_sel = sel.transform(prep.transform(X_train_raw))
sel_names = np.array(prep.get_feature_names_out())[sel.get_support()]
sample_idx = np.random.RandomState(42).choice(X_sel.shape[0], size=min(200, X_sel.shape[0]), replace=False)
X_shap = pd.DataFrame(X_sel[sample_idx], columns=sel_names)
if clf.__class__.__name__ in ['RandomForestClassifier', 'DecisionTreeClassifier', 'XGBClassifier', 'CatBoostClassifier']:
    explainer = shap.TreeExplainer(clf)
    sv = explainer.shap_values(X_shap)
    sv = sv[1] if isinstance(sv, list) else sv
else:
    explainer = shap.Explainer(clf, X_shap)
    sv = explainer(X_shap).values
    sv = sv[1] if isinstance(sv, list) else sv
shap.summary_plot(sv, X_shap, show=False)
plt.tight_layout(); plt.show()
shap.summary_plot(sv, X_shap, plot_type='bar', show=False)
plt.tight_layout(); plt.show()

env_models = Path('models')
env_models.mkdir(exist_ok=True)
model_path = env_models / 'employee_attrition_model.pkl'
joblib.dump(final_pipe, model_path)
loaded_model = joblib.load(model_path)
sample_rows = X_test_raw.sample(n=min(5, len(X_test_raw)), random_state=42)
demo = pd.DataFrame({
    'Actual_Attrition': y_test.loc[sample_rows.index].map({0: 'No', 1: 'Yes'}),
    'Predicted_Attrition': pd.Series(loaded_model.predict(sample_rows), index=sample_rows.index).map({0: 'No', 1: 'Yes'}),
    'Probability_Attrition_Yes': np.round(proba(loaded_model, sample_rows), 4)
})
print(f'Model saved to: {model_path.resolve()}')
display(demo)

## Section 18 - Conclusion

This notebook delivered a portable employee attrition classification workflow with cleaning, analysis, feature engineering, leakage-safe modeling, tuning, explainability, and a saved deployment artifact.